In [25]:
!pip install google-genai sentence-transformers faiss-cpu requests beautifulsoup4 -q

from google import genai
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(model="gemini-3.5-flash", contents="test prompt")
print(response.text)

Test successful! I am online and ready to help. How can I assist you today?


In [11]:
!pip install pdfplumber -q

import pdfplumber
from google.colab import files

print("Upload your PDF files:")
uploaded = files.upload()
docs = []  # everything gets collected here, from all sources

def load_pdfs(uploaded_files):
    loaded = []
    for fname in uploaded_files:
        if fname.endswith(".pdf"):
            text = ""
            with pdfplumber.open(fname) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
            loaded.append({"source": fname, "text": text, "type": "pdf"})
    return loaded

docs.extend(load_pdfs(uploaded))
print(f"After PDF load: {len(docs)} docs")

# sanity check
print(docs[-1]["text"][:500])

Upload your PDF files:


Saving InstituteBrochure.pdf to InstituteBrochure (2).pdf
After PDF load: 1 docs



In [12]:
def scrape_page(url, source_name):
    resp = requests.get(url, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    text = "\n".join(line.strip() for line in soup.get_text(separator="\n").split("\n") if line.strip())
    return {"source": source_name, "text": text, "type": "web"}

club_urls = [
    ("https://itc.gymkhana.iitb.ac.in/wncc/", "WnCC"),
    ("https://krittikaiitb.github.io/", "Kritika"),
    ("https://itc.gymkhana.iitb.ac.in/energyclub/#/homepage", "Energy"),
    ("https://erc.tech-iitb.org/", "Electronics and Robotics club"),
    ("https://chemclubiitb.wordpress.com/","Chemistry club"),
    ("https://sites.google.com/view/biox-iitb/home", "BioX"),
    ("https://itc.gymkhana.iitb.ac.in/aero/", "Aeromodelling club"),
    ("https://mnp-club.github.io/","Maths and Physics Club"),
    ("https://cseciitb.github.io/", "Cyber Security club"),
    ("https://www.iitbracing.org/","IITB racing team"),
    ("https://www.auv-iitb.org/", "AUV"),
    ("https://iitbmartian.github.io/", "MRT"),
    ("https://iitbrocketteam.in/", "Rocket Team"),
    ("https://www.aero.iitb.ac.in/satlab/", "SSP"),
    ("https://rakshakiitb.in/", "Rakshak")
]

for url, name in club_urls:
    docs.append(scrape_page(url, name))

print(f"After scraping: {len(docs)} docs")

# sanity check — confirm nothing came back empty/garbled
print(docs[-1]["text"][:500])

After scraping: 16 docs
Home | Team Rakshak IIT Bombay
A mile of highway will take you a mile
But
A mile of runway can take you anywhere
Rakshak IITB is the official team of IIT Bombay consisting of 40+ students from various disciplines who work collaboratively to develop a fleet of robust Unmanned Aerial Vehicles (UAVs) to support Search and Rescue Operations (SRO) in the event of a disaster. The team was formed in the year 2015 by some of the IITB enthusiasts to build drones/planes that can be used for rescue mission


In [13]:
def chunk_markdown(text, source):
    sections = re.split(r'\n(?=## )', text)
    return [{"text": s.strip(), "source": source} for s in sections if s.strip()]

def chunk_plain(text, source, chunk_size=800, overlap=100):
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        c = text[i:i+chunk_size].strip()
        if c:
            chunks.append({"text": c, "source": source})
    return chunks

all_chunks = []
for doc in docs:
    if doc["type"] == "markdown":
        all_chunks.extend(chunk_markdown(doc["text"], doc["source"]))
    else:  # "web" or "pdf" — both plain text, same treatment
        all_chunks.extend(chunk_plain(doc["text"], doc["source"]))

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 46


In [16]:
from google import genai
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests
from bs4 import BeautifulSoup
import os
import re

embedder = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True).astype("float32")
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"Indexed {index.ntotal} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Indexed 46 chunks


In [17]:
def retrieve(query, k=4):
    q_vec = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, k)
    return [all_chunks[i] for i in idxs[0]]

In [18]:
def generate_answer(prompt: str) -> str:
    try:
        response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
        return response.text.strip()
    except Exception as e:
        return f"[Generation failed: {e}]"

In [22]:
def ask(query, k=4):
    retrieved = retrieve(query, k)
    context = "\n\n".join(f"[Source: {c['source']}]\n{c['text']}" for c in retrieved)
    prompt = f"""You are an assistant that answers questions about WnCC at IIT Bombay.
Answer ONLY using the context below. If the answer isn't in the context, say "I don't know".

Context:
{context}

Question: {query}
Answer:"""
    answer = generate_answer(prompt)
    sources = list(set(c["source"] for c in retrieved))
    return answer, sources

In [23]:
answer, sources = ask("What is WnCC?")
print(answer)
print("Sources:", sources)

Based on the provided context, the Web and Coding Club (WnCC) is one of the biggest clubs of IIT Bombay. As part of the Institute Technical Council, its purpose is to:
* Provide a gateway for people in the institute to join the coding community.
* Create a platform that allows students to gain assistance and mentorship to enhance their coding ability.
* Provide every student at IIT Bombay with an opportunity and the right start to learn how to code and develop a passion for it.
Sources: ['WnCC']
